# 02 - Why naive inversion fails

Lecture section: 3.1-3.5  |  Spine term this tutorial changes: the prior **R** (here we use **none**, R = 0)

$$\hat{x} = \arg\min_x\ \underbrace{D(Ax, y)}_{\mathrm{data\ fidelity}} + \underbrace{R(x)}_{\mathrm{prior}}$$

Before we add any prior, let us see what happens with **pure data-fidelity** inversion:
$R = 0$, so we just try to undo $A$ and match $y$. Our physics $D$ is a **sparse-view, noisy CT**
operator (our PAT/wave stand-in): few angles makes $A$ ill-posed, and a little measurement noise
is enough to make the naive inverse **explode**. This is the motivation for the entire rest of
the lecture: we *need* a prior $R$ to tame the inversion.

In [1]:
import tutorial_common as tc
import deepinv as dinv
import torch

tc.set_seed()

deepinv 0.4.1 | torch 2.9.1 | device cpu


## Step 1 - One ill-posed problem, two measurements (clean vs noisy)

We fix the physics $D$: a **20-angle** CT operator. The clean operator gives us the noiseless
sinogram $y_{\text{clean}} = Ax$; the noisy operator adds Gaussian noise of std $\sigma = 0.05$
to produce $y_{\text{noisy}}$. Same object $x$, same forward map $A$ - the only difference is a
whisper of noise on the measurements.

In [2]:
x = tc.load_hero(128)                                  # (1,1,128,128) phantom in [0,1]
phys_clean = tc.ct_physics(angles=20, sigma=0.0)       # noiseless physics
phys_noisy = tc.ct_physics(angles=20, sigma=0.05)      # same A, + measurement noise

y_clean = phys_clean.A(x)        # A(x): deterministic, clean sinogram
y_noisy = phys_noisy(x)          # physics(x): applies the noise model
print(f"sinogram shape {tuple(y_clean.shape)}  (detector cells x angles)")
print(f"measurement noise std added: {(y_noisy - y_clean).std().item():.3f}")

sinogram shape (1, 1, 182, 20)  (detector cells x angles)
measurement noise std added: 0.050


## Step 2 - Naive inversion: noise amplification

With $R = 0$, "solving" the inverse problem means inverting $A$ to fit $y$. Two ways:

- **FBP** (filtered back-projection): the classical fast inverse for CT.
- **$A^\dagger$** (the least-squares pseudo-inverse): minimizes $\|Ax - y\|^2$ exactly - the
  purest possible "just fit the data, no prior" reconstruction.

On the **clean** data FBP is imperfect (only 20 angles) but recognizable. The instant we feed it
**noisy** data the same operator amplifies that noise; and $A^\dagger$ - which trusts the data
completely - is **destroyed**, producing values far outside the $[0,1]$ range of the object.

In [3]:
fbp_clean = phys_clean.fbp(y_clean)        # filtered back-projection, clean data
fbp_noisy = phys_clean.fbp(y_noisy)        # same FBP, noisy data
adag_noisy = phys_clean.A_dagger(y_noisy)  # least-squares pseudo-inverse, noisy data (no prior!)

print(f"object x         range [{x.min():.2f}, {x.max():.2f}]")
print(f"A_dagger(y_noisy) range [{adag_noisy.min():.2f}, {adag_noisy.max():.2f}]  <- blown up!")

tc.save_images(
    [x, fbp_clean, fbp_noisy, adag_noisy],
    titles=[
        "x (object)",
        tc.title_psnr("FBP(y_clean)", fbp_clean, x),
        tc.title_psnr("FBP(y_noisy)", fbp_noisy, x),
        tc.title_psnr("A_dagger(y_noisy)", adag_noisy, x),
    ],
    fname="02_naive_recon.png",
    suptitle="No prior (R = 0): a little noise wrecks the naive inverse",
)

object x         range [0.00, 1.00]
A_dagger(y_noisy) range [-5.77, 4.43]  <- blown up!


saved /Users/jonathan/Code/deepinv/lecture-tutorials/figures/02_naive_recon.png


The pseudo-inverse went from $[0,1]$ to wildly negative and super-bright values, and its PSNR is
the *worst* of all - precisely because it trusts the noisy data most. Why? Because $A$ has tiny
singular values, and inverting divides by them. Let us measure that.

## Step 3a - A headline condition number

The **condition number** $\kappa = \sigma_{\max}/\sigma_{\min}$ tells us how much $A$ stretches
inputs relative to how much it *shrinks* others; large $\kappa$ means inversion massively amplifies
the directions $A$ barely sees. We compute it at a smaller 64px size (it is slow). This is only a
**power-iteration estimate**: it nails $\sigma_{\max}$ but badly *under*-estimates the tiny
$\sigma_{\min}$, so the headline number below looks deceptively mild. The honest measure comes from
the full **spectrum** next.

In [4]:
cn = tc.ct_physics(angles=20, size=64).condition_number(tc.load_hero(64))
print(f"condition number (estimate, 64px, 20 angles): {float(cn):.1f}")

condition number (estimate, 64px, 20 angles): 83.0


## Step 3b - The centerpiece: the singular value spectrum

To see *why* inversion explodes, we build the operator $A$ **explicitly** as a matrix (small size
$N = 32$, 30 angles), then take its SVD $A = U \Sigma V^\top$. We probe one column at a time by
applying $A$ to each unit pixel $e_i$ - the result is the $i$-th column of $A$.

We need this matrix-form $A$ once, purely to *diagnose* the problem; the reconstructions above used
the fast operators. The true condition number $\sigma_{\max}/\sigma_{\min}$ printed below is about
**$10^7$** -- five orders of magnitude worse than the power-iteration estimate above, and the real
fingerprint of an ill-posed problem.

In [5]:
N, n_angles, sigma = 32, 30, 0.05
p = tc.ct_physics(angles=n_angles, size=N)

# Build A column by column: A e_i = i-th column of A.
cols = []
for i in range(N * N):
    e = torch.zeros(1, 1, N, N, device=tc.DEVICE)
    e.view(-1)[i] = 1.0
    cols.append(p.A(e).reshape(-1))
A_mat = torch.stack(cols, dim=1)                 # (n_measurements, N*N)
U, S, Vh = torch.linalg.svd(A_mat, full_matrices=False)
print(f"A_mat {tuple(A_mat.shape)} | sigma_max={S[0]:.2e}  sigma_min={S[-1]:.2e}  ratio={S[0]/S[-1]:.1e}")

A_mat (1380, 1024) | sigma_max=1.00e+00  sigma_min=8.32e-08  ratio=1.2e+07


## Step 3c - The Picard plot: noise / tiny singular value = explosion

The least-squares solution is $\hat{x} = \sum_i \frac{u_i^\top y}{\sigma_i}\, v_i$. So each
measurement coefficient $u_i^\top y$ gets **divided by** the singular value $\sigma_i$.

- The **clean** signal's coefficients $|u_i^\top y|$ *decay* alongside $\sigma_i$ (the Picard
  condition) - the true object lives in the well-seen directions.
- **Noise** does *not* decay: at large $i$, $|u_i^\top y|$ is dominated by noise of size $\sim\sigma$,
  and dividing by a tiny $\sigma_i$ sends $|u_i^\top y| / \sigma_i$ to enormous values. Those blown-up
  terms are what destroyed $A^\dagger$ above.

We build $y$ at the same small size, add the same noise, and plot the spectrum and the Picard
coefficients.

In [6]:
x_small = tc.load_hero(N).reshape(-1)
y_small_clean = A_mat @ x_small
tc.set_seed()                                    # reproducible noise
y_small = y_small_clean + sigma * torch.randn_like(y_small_clean)

coeff = (U.t() @ y_small).abs()                  # |u_i^T y|        (noisy measurement coefficients)
picard = coeff / S                               # |u_i^T y| / sigma_i  (the division that explodes)
idx = torch.arange(S.numel())

# Left: singular values (log-y) decaying to ~0.  Right: Picard plot showing the blow-up.
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].semilogy(idx, S, color=tc.PALETTE["A"], lw=2)
ax[0].set(title="Singular values $\\sigma_i$ decay to ~0", xlabel="index $i$", ylabel="$\\sigma_i$")
ax[1].semilogy(idx, coeff, color=tc.PALETTE["y"], lw=2, label="$|u_i^\\top y|$  (decays)")
ax[1].semilogy(idx, picard, color=tc.PALETTE["x"], lw=2, label="$|u_i^\\top y| / \\sigma_i$  (explodes)")
ax[1].set(title="Picard plot: noise / tiny $\\sigma_i$ blows up", xlabel="index $i$", ylabel="magnitude")
ax[1].legend()
fig.suptitle("Why $A^\\dagger$ explodes: dividing measurements (incl. noise) by tiny singular values")
fig.tight_layout()
fig.savefig(str(tc.FIG_DIR / "02_singular_values.png"), dpi=150)
plt.close(fig)
print("saved", str(tc.FIG_DIR / "02_singular_values.png"))
print(f"|u^T y| at last index: {coeff[-1]:.1e}   ->   /sigma_i: {picard[-1]:.1e}  (blown up)")

saved /Users/jonathan/Code/deepinv/lecture-tutorials/figures/02_singular_values.png
|u^T y| at last index: 2.5e-02   ->   /sigma_i: 3.0e+05  (blown up)


## Takeaway

The operator $A$ has singular values that decay to ~0, and the naive (least-squares) inverse
divides the measurements - **noise included** - by those tiny values, so the reconstruction
explodes. Pure data-fidelity ($R = 0$) is not enough: we need a **prior $R$** to regularize the
inversion. That is what the rest of the lecture builds, one smarter $R$ at a time.